In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================

!pip install torch torchvision timm tqdm scikit-learn seaborn

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# ============================================================
# ADD EXPERIMENT BASE FOLDER
# ============================================================

BASE_DIR = "/content/drive/MyDrive/Research_Implementation/Deepfake_SGAM_Experiments"

if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

print("Base SGAM experiment directory ready:")
print(BASE_DIR)

In [ ]:
# ============================================================
# CREATE UNIQUE FOLDER FOR EACH RUN
# BEFORE TRAINING, DEFINE THE EXPERIMENT NAME
# ============================================================

EXPERIMENT_NAME = "Experiment_3_LR=1e-4_BS=16"

EXP_PATH = os.path.join(BASE_DIR, EXPERIMENT_NAME)

os.makedirs(EXP_PATH, exist_ok=True)

print("Running:", EXPERIMENT_NAME)
print("Experiment path:", EXP_PATH)

In [ ]:
# ============================================================
# DEVICE CONFIGURATION
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ============================================================
# DATASET PATHS
# Same structure as your CNN implementation
# ============================================================

train_dir = "/content/drive/MyDrive/Research_Implementation/dataset/train"
val_dir   = "/content/drive/MyDrive/Research_Implementation/dataset/val"
test_dir  = "/content/drive/MyDrive/Research_Implementation/dataset/test"

print("Train path:", train_dir)
print("Val path:", val_dir)
print("Test path:", test_dir)

In [ ]:
# ============================================================
# IMAGE PREPROCESSING
# According to Chapter 3:
# Resize 224x224, ImageNet normalization,
# augmentation only for training
# ============================================================

IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

train_data = datasets.ImageFolder(train_dir, transform=train_transform)
val_data   = datasets.ImageFolder(val_dir, transform=test_transform)
test_data  = datasets.ImageFolder(test_dir, transform=test_transform)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Train samples:", len(train_data))
print("Val samples:", len(val_data))
print("Test samples:", len(test_data))

In [ ]:
# ============================================================
# CHECK CLASS MAPPING
# Important: fake/real index must be clear for evaluation
# ============================================================

print("Train:", train_data.class_to_idx)
print("Val:", val_data.class_to_idx)
print("Test:", test_data.class_to_idx)

num_classes = len(train_data.classes)
print("Number of classes:", num_classes)
print("Classes:", train_data.classes)

## STEP 2: BUILD SpFD-CNN FEATURE EXTRACTOR
## EFFICIENTNET-B0 SPATIAL FEATURE DETECTOR

In [ ]:
# ============================================================
# STEP 2: BUILD SpFD-CNN FEATURE EXTRACTOR
# Extract [320 × 7 × 7] activation maps
# ============================================================

import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights

class SpFDCNNFeatureExtractor(nn.Module):
    def __init__(self):
        super(SpFDCNNFeatureExtractor, self).__init__()

        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        efficientnet = models.efficientnet_b0(weights=weights)

        # Remove final Conv1x1 expansion layer
        # Keep only up to final MBConv block
        self.features = nn.Sequential(
            *list(efficientnet.features.children())[:-1]
        )

    def forward(self, x):

        spatial_activation_maps = self.features(x)

        return spatial_activation_maps

In [ ]:
# ============================================================
# INITIALIZE SpFD-CNN FEATURE EXTRACTOR
# ============================================================

spfd_cnn = SpFDCNNFeatureExtractor().to(device)

print(spfd_cnn)

In [ ]:
# ============================================================
# VERIFY SPATIAL ACTIVATION MAP SHAPE
# Chapter 3 mentions [320, 7, 7]
# ============================================================

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    activation_maps = spfd_cnn(images)

print("Input image batch shape:", images.shape)
print("Spatial activation maps shape:", activation_maps.shape)

##STEP 3: Spatial Guided Attention Mechanism (SGAM) Implementation

In [ ]:
# ============================================================
# STEP 3: SPATIAL-GUIDED ATTENTION MECHANISM (SGAM)
# ============================================================

class SGAModule(nn.Module):

    def __init__(self):
        super(SGAModule, self).__init__()

        # Learnable projection Φ : 196 → 196
        self.projection = nn.Linear(196, 196)

    def forward(self, activation_maps):

        # ----------------------------------------------------
        # activation_maps shape:
        # [batch_size, 320, 7, 7]
        # ----------------------------------------------------

        # Step 1: Channel averaging
        # Output: [batch_size, 7, 7]

        spatial_map = activation_maps.mean(dim=1)

        # ----------------------------------------------------

        # Step 2: Upsample to 14×14
        # Output: [batch_size, 1, 14, 14]

        spatial_map = spatial_map.unsqueeze(1)

        upsampled_map = F.interpolate(
            spatial_map,
            size=(14, 14),
            mode='bilinear',
            align_corners=False
        )

        # ----------------------------------------------------

        # Step 3: Flatten
        # Output: [batch_size, 196]

        flattened_map = upsampled_map.view(
            upsampled_map.size(0),
            -1
        )

        # ----------------------------------------------------

        # Step 4: Linear projection Φ
        # Output: [batch_size, 196]

        attention_bias = self.projection(flattened_map)

        return attention_bias

In [ ]:
# ============================================================
# INITIALIZE SGAM
# ============================================================

sgam = SGAModule().to(device)

print(sgam)

In [ ]:
# ============================================================
# VERIFY SGAM OUTPUT SHAPE
# ============================================================

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():

    activation_maps = spfd_cnn(images)

    attention_bias = sgam(activation_maps)

print("Activation maps shape:", activation_maps.shape)

print("Attention bias shape:", attention_bias.shape)

## STEP 4: Custom DeiT with Attention Bias Injection.

In [ ]:
# ============================================================
# STEP 4: CUSTOM DeiT-SMALL WITH ATTENTION BIAS INJECTION
# ============================================================

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Guided Attention Wrapper
# ============================================================
class GuidedAttentionWrapper(nn.Module):
    def __init__(self, original_attn):
        super(GuidedAttentionWrapper, self).__init__()

        self.original_attn = original_attn
        self.current_bias = None

    def set_attention_bias(self, attention_bias):
        self.current_bias = attention_bias

    def clear_attention_bias(self):
        self.current_bias = None

    def forward(self, x, *args, **kwargs):
        """
        x shape: [batch_size, 197, 384]
        *args and **kwargs are accepted for compatibility with timm.
        """

        B, N, C = x.shape

        qkv = self.original_attn.qkv(x)
        qkv = qkv.reshape(
            B,
            N,
            3,
            self.original_attn.num_heads,
            C // self.original_attn.num_heads
        ).permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.original_attn.scale

        attention_logits = q @ k.transpose(-2, -1)

        # Inject SGAM bias into patch-to-patch attention only
        if self.current_bias is not None:
            bias = self.current_bias.unsqueeze(1).unsqueeze(2)
            attention_logits[:, :, 1:, 1:] = attention_logits[:, :, 1:, 1:] + bias

        attention = attention_logits.softmax(dim=-1)
        attention = self.original_attn.attn_drop(attention)

        x = attention @ v
        x = x.transpose(1, 2).reshape(B, N, C)

        x = self.original_attn.proj(x)
        x = self.original_attn.proj_drop(x)

        return x


In [ ]:
# ============================================================
# DeiT-Small Semantic Feature Detector with SGAM Bias Injection
# ============================================================

class GuidedDeiTSmall(nn.Module):
    def __init__(self):
        super(GuidedDeiTSmall, self).__init__()

        # Load pretrained DeiT-Small
        self.deit = timm.create_model(
            "deit_small_patch16_224",
            pretrained=True
        )

        # Remove original classifier head
        self.deit.head = nn.Identity()

        # Replace first transformer block attention with guided attention
        original_attn = self.deit.blocks[0].attn
        self.deit.blocks[0].attn = GuidedAttentionWrapper(original_attn)

    def forward(self, x, attention_bias):
        """
        x: input image batch [B, 3, 224, 224]
        attention_bias: SGAM output [B, 196]
        """

        # Set attention bias before forward pass
        self.deit.blocks[0].attn.set_attention_bias(attention_bias)

        # Forward pass through DeiT
        cls_embedding = self.deit(x)

        # Clear attention bias after forward pass
        self.deit.blocks[0].attn.clear_attention_bias()

        return cls_embedding

In [ ]:
# ============================================================
# INITIALIZE GUIDED DeiT-SMALL
# ============================================================

guided_deit = GuidedDeiTSmall().to(device)

print("Guided DeiT-Small loaded successfully.")

This completes the main novelty part:



1.   CNN spatial activation maps
2.   SGAM bias
3.   Injected into DeiT attention logits
4.    Guided CLS token



In [ ]:
# ============================================================
# VERIFY GUIDED DeiT OUTPUT SHAPE
# Expected: [batch_size, 384]
# ============================================================

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    activation_maps = spfd_cnn(images)
    attention_bias = sgam(activation_maps)
    guided_cls_token = guided_deit(images, attention_bias)

print("Input image shape:", images.shape)
print("Activation maps shape:", activation_maps.shape)
print("Attention bias shape:", attention_bias.shape)
print("Guided CLS token shape:", guided_cls_token.shape)

STEP 5: BUILDING THE FULL PROPOSED MODEL WITH CLASSIFIER



1. Input Image
2. SpFD-CNN
3. Spatial Activation Maps[320 × 7 × 7]
4. SGAM
5. Attention Bias [196]
6. Guided DeiT-Small
7. Guided CLS Token [384]
8. Classifier
9. Real/Fake Logit  



In [ ]:
# ============================================================
# STEP 5: FULL PROPOSED SGAM MODEL WITH CLASSIFIER
# ============================================================

class ProposedSGAMDeepfakeDetector(nn.Module):
    def __init__(self):
        super(ProposedSGAMDeepfakeDetector, self).__init__()

        # Spatial Feature Detector CNN
        self.spfd_cnn = SpFDCNNFeatureExtractor()

        # Spatial-Guided Attention Mechanism
        self.sgam = SGAModule()

        # Guided DeiT-Small Semantic Feature Detector
        self.guided_deit = GuidedDeiTSmall()

        # Final classifier
        self.classifier = nn.Linear(384, 1)

    def forward(self, x):
        # CNN spatial activation maps
        activation_maps = self.spfd_cnn(x)

        # SGAM attention bias
        attention_bias = self.sgam(activation_maps)

        # Guided CLS token from DeiT
        guided_cls_token = self.guided_deit(x, attention_bias)

        # Final logit
        logit = self.classifier(guided_cls_token)

        return logit

In [ ]:
# ============================================================
# INITIALIZE FULL PROPOSED MODEL
# ============================================================

model = ProposedSGAMDeepfakeDetector().to(device)

print("Full Proposed SGAM Model initialized successfully.")

In [ ]:
# ============================================================
# VERIFY FULL MODEL OUTPUT SHAPE
# Expected output: [batch_size, 1]
# ============================================================

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input image shape:", images.shape)
print("Model output shape:", outputs.shape)

STEP 6: DEFINE LOSS, OPTIMIZER, SCHEDULER, AND TRANING CONFIGURATION

In [ ]:
# ============================================================
# STEP 6: LOSS, OPTIMIZER, SCHEDULER, TRAINING CONFIGURATION
# ============================================================

import torch.optim as optim

# -----------------------------
# Training Configuration
# -----------------------------

LEARNING_RATE = 0.0001
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5

# -----------------------------
# Loss Function
# -----------------------------
# Model outputs logits, so use BCEWithLogitsLoss.
# Sigmoid is applied only during evaluation/inference.

criterion = nn.BCEWithLogitsLoss()

# -----------------------------
# Optimizer
# -----------------------------

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# -----------------------------
# Learning Rate Scheduler
# -----------------------------

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3
)

print("Training configuration ready.")
print("Learning Rate:", LEARNING_RATE)
print("Max Epochs:", MAX_EPOCHS)
print("Early Stopping Patience:", EARLY_STOPPING_PATIENCE)
print("Loss Function: BCEWithLogitsLoss")
print("Optimizer: Adam")
print("Scheduler: ReduceLROnPlateau")

In [ ]:
# ============================================================
# SAVE TRAINING CONFIGURATION
# ============================================================

config_path = os.path.join(EXP_PATH, "training_config.txt")

with open(config_path, "w") as f:
    f.write("Experiment Name: " + EXPERIMENT_NAME + "\n")
    f.write("Model: Proposed SGAM Deepfake Detector\n")
    f.write("Architecture: SpFD-CNN + SGAM + Guided DeiT-Small + Classifier\n")
    f.write("Spatial activation maps: [320, 7, 7]\n")
    f.write("Attention bias: [196]\n")
    f.write("Guided CLS token: [384]\n")
    f.write("Loss Function: BCEWithLogitsLoss\n")
    f.write("Optimizer: Adam\n")
    f.write(f"Learning Rate: {LEARNING_RATE}\n")
    f.write(f"Max Epochs: {MAX_EPOCHS}\n")
    f.write(f"Batch Size: {BATCH_SIZE}\n")
    f.write("Scheduler: ReduceLROnPlateau, factor=0.1, patience=3\n")
    f.write(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE}\n")

print("Training configuration saved to:")
print(config_path)

# Step 7 — Training and Validation Loop with automatic saving of model, CSV history, and curves.

In [ ]:
# ============================================================
# STEP 7: TRAINING AND VALIDATION LOOP
# Saves:
# - best_model.pth
# - training_history.csv
# - loss_curve.png
# - accuracy_curve.png
# ============================================================

best_val_loss = float("inf")
early_stop_counter = 0

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

best_model_path = os.path.join(EXP_PATH, "best_sgam_model.pth")
history_csv_path = os.path.join(EXP_PATH, "training_history.csv")

for epoch in range(MAX_EPOCHS):

    # ============================
    # Training Phase
    # ============================

    model.train()

    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Training]")

    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

        train_bar.set_postfix({
            "loss": loss.item()
        })

    train_loss = running_train_loss / total_train
    train_accuracy = correct_train / total_train

    # ============================
    # Validation Phase
    # ============================

    model.eval()

    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Validation]")

        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            running_val_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

            val_bar.set_postfix({
                "loss": loss.item()
            })

    val_loss = running_val_loss / total_val
    val_accuracy = correct_val / total_val

    # ============================
    # Scheduler
    # ============================

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # ============================
    # Save History
    # ============================

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["learning_rate"].append(current_lr)

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_csv_path, index=False)

    print(f"\nEpoch [{epoch+1}/{MAX_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_accuracy:.4f}")
    print(f"Learning Rate: {current_lr}")

    # ============================
    # Save Best Model
    # ============================

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "class_to_idx": train_data.class_to_idx
        }, best_model_path)

        print("Best model saved.")

    else:
        early_stop_counter += 1
        print(f"No improvement. Early stopping counter: {early_stop_counter}/{EARLY_STOPPING_PATIENCE}")

    # ============================
    # Early Stopping
    # ============================

    if early_stop_counter >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

print("Training completed.")
print("Best validation loss:", best_val_loss)

PLOT TRAINING AND VALIDATION LEARNING CURVES

In [ ]:
# ============================================================
# PLOT TRAINING AND VALIDATION LEARNING CURVES
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

# Use your Experiment 1 path
EXP_PATH = "/content/drive/MyDrive/Research_Implementation/Deepfake_SGAM_Experiments/Experiment_1_LR=1e-4_BS=32"

history_csv_path = os.path.join(EXP_PATH, "training_history.csv")

history_df = pd.read_csv(history_csv_path)

print(history_df.head())

# ============================================================
# LOSS CURVE
# ============================================================

plt.figure(figsize=(8, 6))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curve")
plt.legend()
plt.grid(True)

loss_curve_path = os.path.join(EXP_PATH, "training_validation_loss_curve.png")
plt.savefig(loss_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("Loss curve saved at:", loss_curve_path)

# ============================================================
# ACCURACY CURVE
# ============================================================

plt.figure(figsize=(8, 6))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Training Accuracy")
plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy Curve")
plt.legend()
plt.grid(True)

accuracy_curve_path = os.path.join(EXP_PATH, "training_validation_accuracy_curve.png")
plt.savefig(accuracy_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("Accuracy curve saved at:", accuracy_curve_path)

## Step 8 — Test Evaluation + Save Final Results

1. test_metrics.csv
2. classification_report.txt
3. predictions.csv
4. confusion_matrix.png
5. roc_curve.png
6. probability_distribution.png


In [ ]:
# ============================================================
# STEP 8: TEST EVALUATION
# Saves metrics, predictions, confusion matrix, ROC curve
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# -----------------------------
# Load Best Model
# -----------------------------

best_model_path = os.path.join(EXP_PATH, "best_sgam_model.pth")

checkpoint = torch.load(best_model_path, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("Best model loaded successfully.")

# -----------------------------
# Testing
# -----------------------------

all_labels = []
all_probs = []
all_preds = []
all_paths = []

with torch.no_grad():
    test_bar = tqdm(test_loader, desc="Testing")

    for images, labels in test_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())

# -----------------------------
# Metrics
# -----------------------------

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc_score = roc_auc_score(all_labels, all_probs)

print("Test Accuracy:", accuracy)
print("Test Precision:", precision)
print("Test Recall:", recall)
print("Test F1 Score:", f1)
print("Test AUC:", auc_score)

# -----------------------------
# Save Metrics CSV
# -----------------------------

metrics_df = pd.DataFrame([{
    "model": "Proposed_SGAM",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "auc_roc": auc_score
}])

metrics_path = os.path.join(EXP_PATH, "test_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Metrics saved:", metrics_path)

# -----------------------------
# Save Classification Report
# -----------------------------

report = classification_report(
    all_labels,
    all_preds,
    target_names=test_data.classes
)

report_path = os.path.join(EXP_PATH, "classification_report.txt")

with open(report_path, "w") as f:
    f.write(report)

print("Classification report saved:", report_path)

# -----------------------------
# Save Predictions CSV
# -----------------------------

predictions_df = pd.DataFrame({
    "true_label": all_labels,
    "predicted_label": all_preds,
    "probability_fake": all_probs
})

predictions_path = os.path.join(EXP_PATH, "predictions.csv")
predictions_df.to_csv(predictions_path, index=False)

print("Predictions saved:", predictions_path)

# -----------------------------
# Confusion Matrix
# -----------------------------

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_data.classes,
    yticklabels=test_data.classes
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Proposed SGAM Model")

cm_path = os.path.join(EXP_PATH, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("Confusion matrix saved:", cm_path)

# -----------------------------
# ROC Curve
# -----------------------------

fpr, tpr, thresholds = roc_curve(all_labels, all_probs)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Proposed SGAM Model")
plt.legend()
plt.grid(True)

roc_path = os.path.join(EXP_PATH, "roc_curve.png")
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

print("ROC curve saved:", roc_path)

# -----------------------------
# Probability Distribution
# -----------------------------

plt.figure(figsize=(8, 6))
plt.hist(
    [p for p, y in zip(all_probs, all_labels) if y == 0],
    bins=30,
    alpha=0.6,
    label="Real"
)
plt.hist(
    [p for p, y in zip(all_probs, all_labels) if y == 1],
    bins=30,
    alpha=0.6,
    label="Fake"
)
plt.xlabel("Predicted Probability of Fake")
plt.ylabel("Frequency")
plt.title("Probability Distribution - Proposed SGAM Model")
plt.legend()
plt.grid(True)

prob_path = os.path.join(EXP_PATH, "probability_distribution.png")
plt.savefig(prob_path, dpi=300, bbox_inches="tight")
plt.show()

print("Probability distribution saved:", prob_path)

### Step 9 — Visual Explanation of SGAM Attention Flow

Image → CNN activation map → SGAM heatmap → DeiT guidance

In [ ]:
# ============================================================
# STEP 9: VISUAL EXPLANATION OF SGAM ATTENTION FLOW
# ============================================================

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import os

model.eval()

# Save folder
VIS_DIR = os.path.join(EXP_PATH, "SGAM_Attention_Visualizations")
os.makedirs(VIS_DIR, exist_ok=True)

# ImageNet mean/std for reverse normalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img.permute(1, 2, 0).numpy()

def normalize_map(x):
    x = x - x.min()
    x = x / (x.max() + 1e-8)
    return x

def visualize_sgam_flow(model, data_loader, num_images=6):
    count = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.no_grad():

            # 1. CNN spatial activation maps: [B, 320, 7, 7]
            activation_maps = model.spfd_cnn(images)

            # 2. Channel average: [B, 7, 7]
            spatial_maps_7x7 = activation_maps.mean(dim=1)

            # 3. Upsample to 14x14
            spatial_maps_14x14 = F.interpolate(
                spatial_maps_7x7.unsqueeze(1),
                size=(14, 14),
                mode="bilinear",
                align_corners=False
            ).squeeze(1)

            # 4. SGAM projected attention bias: [B, 196]
            attention_bias = model.sgam(activation_maps)

            # 5. Reshape attention bias to 14x14
            attention_bias_maps = attention_bias.view(-1, 14, 14)

            # 6. Final prediction
            logits = model(images)
            probs = torch.sigmoid(logits).squeeze(1)
            preds = (probs >= 0.5).long()

        batch_size = images.size(0)

        for i in range(batch_size):
            if count >= num_images:
                return

            original_img = denormalize(images[i])

            cnn_map = spatial_maps_7x7[i].detach().cpu().numpy()
            cnn_map = normalize_map(cnn_map)

            sgam_map = spatial_maps_14x14[i].detach().cpu().numpy()
            sgam_map = normalize_map(sgam_map)

            bias_map = attention_bias_maps[i].detach().cpu().numpy()
            bias_map = normalize_map(bias_map)

            # Resize bias map to 224x224 for overlay
            bias_tensor = torch.tensor(bias_map).unsqueeze(0).unsqueeze(0).float()
            bias_resized = F.interpolate(
                bias_tensor,
                size=(224, 224),
                mode="bilinear",
                align_corners=False
            ).squeeze().numpy()

            true_label = "Fake" if labels[i].item() == 1 else "Real"
            pred_label = "Fake" if preds[i].item() == 1 else "Real"
            prob_fake = probs[i].item()

            plt.figure(figsize=(18, 4))

            # Original image
            plt.subplot(1, 5, 1)
            plt.imshow(original_img)
            plt.title(f"Original Image\nTrue: {true_label}")
            plt.axis("off")

            # CNN activation map
            plt.subplot(1, 5, 2)
            plt.imshow(cnn_map, cmap="jet")
            plt.title("CNN Spatial Map\n7×7")
            plt.axis("off")

            # SGAM upsampled map
            plt.subplot(1, 5, 3)
            plt.imshow(sgam_map, cmap="jet")
            plt.title("SGAM Upsampled Map\n14×14")
            plt.axis("off")

            # Attention bias map
            plt.subplot(1, 5, 4)
            plt.imshow(bias_map, cmap="jet")
            plt.title("Projected Attention Bias\n14×14")
            plt.axis("off")

            # Overlay
            plt.subplot(1, 5, 5)
            plt.imshow(original_img)
            plt.imshow(bias_resized, cmap="jet", alpha=0.45)
            plt.title(f"Bias Overlay\nPred: {pred_label} ({prob_fake:.3f})")
            plt.axis("off")

            #plt.suptitle("Visual Explanation of SGAM Attention Flow", fontsize=14)
            plt.tight_layout()

            save_path = os.path.join(
                VIS_DIR,
                f"sgam_flow_sample_{count+1}_{true_label}_pred_{pred_label}.png"
            )

            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.show()

            count += 1

# Run visualization
visualize_sgam_flow(model, test_loader, num_images=6)

print("SGAM visualizations saved in:")
print(VIS_DIR)

In [ ]:
# ============================================================
# STEP 9B: SGAM VISUALIZATION FOR FAKE IMAGES ONLY
# ============================================================

FAKE_VIS_DIR = os.path.join(EXP_PATH, "SGAM_Fake_Image_Visualizations")
os.makedirs(FAKE_VIS_DIR, exist_ok=True)

def visualize_sgam_fake_images(model, data_loader, num_images=6):
    model.eval()
    count = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Select only fake images
        fake_indices = (labels == 1).nonzero(as_tuple=True)[0]

        if len(fake_indices) == 0:
            continue

        with torch.no_grad():
            activation_maps = model.spfd_cnn(images)
            spatial_maps_7x7 = activation_maps.mean(dim=1)

            spatial_maps_14x14 = F.interpolate(
                spatial_maps_7x7.unsqueeze(1),
                size=(14, 14),
                mode="bilinear",
                align_corners=False
            ).squeeze(1)

            attention_bias = model.sgam(activation_maps)
            attention_bias_maps = attention_bias.view(-1, 14, 14)

            logits = model(images)
            probs = torch.sigmoid(logits).squeeze(1)
            preds = (probs >= 0.5).long()

        for idx in fake_indices:
            if count >= num_images:
                return

            i = idx.item()

            original_img = denormalize(images[i])

            cnn_map = normalize_map(spatial_maps_7x7[i].detach().cpu().numpy())
            sgam_map = normalize_map(spatial_maps_14x14[i].detach().cpu().numpy())
            bias_map = normalize_map(attention_bias_maps[i].detach().cpu().numpy())

            bias_tensor = torch.tensor(bias_map).unsqueeze(0).unsqueeze(0).float()
            bias_resized = F.interpolate(
                bias_tensor,
                size=(224, 224),
                mode="bilinear",
                align_corners=False
            ).squeeze().numpy()

            pred_label = "Fake" if preds[i].item() == 1 else "Real"
            prob_fake = probs[i].item()

            plt.figure(figsize=(18, 4))

            plt.subplot(1, 5, 1)
            plt.imshow(original_img)
            plt.title("Original Fake Image")
            plt.axis("off")

            plt.subplot(1, 5, 2)
            plt.imshow(cnn_map, cmap="jet")
            plt.title("CNN Spatial Map\n7×7")
            plt.axis("off")

            plt.subplot(1, 5, 3)
            plt.imshow(sgam_map, cmap="jet")
            plt.title("Upsampled Spatial Map\n14×14")
            plt.axis("off")

            plt.subplot(1, 5, 4)
            plt.imshow(bias_map, cmap="jet")
            plt.title("SGAM Attention Bias\n14×14")
            plt.axis("off")

            plt.subplot(1, 5, 5)
            plt.imshow(original_img)
            plt.imshow(bias_resized, cmap="jet", alpha=0.45)
            plt.title(f"Bias Overlay\nPred: {pred_label} ({prob_fake:.3f})")
            plt.axis("off")

            #plt.suptitle("SGAM Attention Flow for Fake Image", fontsize=14)
            plt.tight_layout()

            save_path = os.path.join(
                FAKE_VIS_DIR,
                f"fake_sgam_flow_{count+1}_pred_{pred_label}_prob_{prob_fake:.3f}.png"
            )

            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.show()

            count += 1

visualize_sgam_fake_images(model, test_loader, num_images=8)

print("Fake image SGAM visualizations saved in:")
print(FAKE_VIS_DIR)

GRAD-CAM
## New section

In [ ]:
# ============================================================
# LOAD BEST SGAM MODEL USING MANUAL MODEL PATH
# ============================================================

best_model_path = "/content/drive/MyDrive/Research_Implementation/Deepfake_SGAM_Experiments/Experiment_1_LR=1e-4_BS=32/best_sgam_model.pth"

print("Loading model from:")
print(best_model_path)

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

# If saved as checkpoint dictionary
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])

# If directly saved state_dict
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("Best SGAM model loaded successfully.")

In [ ]:
# ============================================================
# SGAM-GUIDED GRAD-CAM VISUALIZATION
# Shows regions contributing to final prediction
# ============================================================

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import os

model.eval()

GRADCAM_DIR = os.path.join(EXP_PATH, "SGAM_GradCAM_Visualizations")
os.makedirs(GRADCAM_DIR, exist_ok=True)

# ImageNet reverse normalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img.permute(1, 2, 0).numpy()

def normalize_map(x):
    x = x - x.min()
    x = x / (x.max() + 1e-8)
    return x

# ------------------------------------------------------------
# Check class mapping
# ------------------------------------------------------------

print("Class mapping:", test_data.class_to_idx)

fake_index = test_data.class_to_idx["fake"]
print("Fake class index:", fake_index)

In [ ]:
# ============================================================
# GRAD-CAM FUNCTION
# ============================================================

def generate_sgam_gradcam(model, image_tensor, target_class_index):
    """
    image_tensor: [1, 3, 224, 224]
    target_class_index: fake class index from ImageFolder
    """

    gradients = []
    activations = []

    # Hook final SpFD-CNN feature layer
    target_layer = model.spfd_cnn.features[-1]

    def forward_hook(module, input, output):
        activations.append(output)

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_full_backward_hook(backward_hook)

    model.zero_grad()

    logits = model(image_tensor)

    # BCE model output:
    # sigmoid(logit) = probability of class label 1
    # If fake index is 1, fake score = logit
    # If fake index is 0, fake score = -logit
    if target_class_index == 1:
        target_score = logits[:, 0]
    else:
        target_score = -logits[:, 0]

    target_score.backward()

    act = activations[0]      # [1, 320, 7, 7]
    grad = gradients[0]       # [1, 320, 7, 7]

    # Global average pooling of gradients
    weights = grad.mean(dim=(2, 3), keepdim=True)

    # Weighted activation maps
    cam = (weights * act).sum(dim=1, keepdim=True)
    cam = F.relu(cam)

    # Resize to image size
    cam = F.interpolate(
        cam,
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    )

    cam = cam.squeeze().detach().cpu().numpy()
    cam = normalize_map(cam)

    forward_handle.remove()
    backward_handle.remove()

    prob = torch.sigmoid(logits).item()

    if target_class_index == 1:
        prob_fake = prob
    else:
        prob_fake = 1 - prob

    return cam, prob_fake

In [ ]:
# ============================================================
# VISUALIZE FAKE IMAGES USING SGAM-GUIDED GRAD-CAM
# ============================================================

def visualize_fake_gradcam(model, data_loader, num_images=5):
    count = 0

    for images, labels in data_loader:
        for i in range(images.size(0)):

            # Select only fake images based on actual class mapping
            if labels[i].item() != fake_index:
                continue

            if count >= num_images:
                return

            image = images[i].unsqueeze(0).to(device)

            cam, prob_fake = generate_sgam_gradcam(
                model,
                image,
                target_class_index=fake_index
            )

            original_img = denormalize(images[i])

            plt.figure(figsize=(12, 4))

            # Original image
            plt.subplot(1, 3, 1)
            plt.imshow(original_img)
            plt.title("Original Fake Image")
            plt.axis("off")

            # Grad-CAM heatmap
            plt.subplot(1, 3, 2)
            plt.imshow(cam, cmap="jet")
            plt.title("SGAM-Guided Grad-CAM")
            plt.axis("off")

            # Overlay
            plt.subplot(1, 3, 3)
            plt.imshow(original_img)
            plt.imshow(cam, cmap="jet", alpha=0.45)
            plt.title(f"Overlay\nPred Fake: {prob_fake:.3f}")
            plt.axis("off")

            plt.tight_layout()

            save_path = os.path.join(
                GRADCAM_DIR,
                f"sgam_gradcam_fake_{count+1}_prob_{prob_fake:.3f}.png"
            )

            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.show()

            count += 1

visualize_fake_gradcam(model, test_loader, num_images=5)

print("SGAM-guided Grad-CAM visualizations saved in:")
print(GRADCAM_DIR)